# Sprint 2 · Track and explain controlled experiments

This notebook is an optional companion to the [Campus source/terminal route](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/README.md). Work in the same clone and locked Python 3.12 environment. It imports reusable core functions, reopens saved decisions and keeps independent cells unsolved. Complete your own attempt before consuming its result or opening feedback. A notebook cannot substitute for actual MLflow UI inspection or an approved-assistant request.


In [ ]:
from IPython import get_ipython
get_ipython().run_line_magic("matplotlib", "inline")
from pathlib import Path
import sys, json, subprocess
import pandas as pd
import numpy as np
from linden_mlops import Config, Workspace, verify_summary
from linden_mlops.lifecycle import ROOT, code_revision
from linden_mlops.project import Project
from linden_mlops.visuals import plot_experiments, save_figure
from IPython.display import display
if sys.version_info[:2] != (3, 12):
    raise RuntimeError("Use the project's locked Python 3.12 environment.")
progress = Project()
state = progress.work
ws = Workspace(progress.tracking)
figures = state / 'figures'
print("Imported application source:", ROOT)
print("Notebook interpreter:", sys.executable)
print("Saved decisions:", sorted(progress.data))
print("Source identity:", code_revision())


## A-S2-01 · Why ad hoc tracking breaks down at scale

A score becomes usable when its measurement, settings and fitted artifact remain connected to one execution ID. Manual copying can break any of those links. Diagnose which decision is blocked before trusting the smallest number.

### Learning outcome

Identify 3+ failure modes of tracking experiments manually (spreadsheets, notebook print statements) once a team runs many models, and explain why systematic tracking matters for auditability.

[Campus explanation, task and post-attempt feedback](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/sprint-2/01-why-ad-hoc-tracking-breaks-down-at-scale/lesson.md).


## A-S2-02 · Controls, confounders, and decision metrics

Ridge learns feature weights while alpha restrains their size. Change this setting while keeping data, training-only scaling, fitting method and validation rows fixed. MAE is the average miss magnitude; RMSE gives a large individual miss more influence. Lower validation rmse_kwh is the declared primary rule; diagnostics and batch completion answer other questions.

### Learning outcome

Explain why a run comparison is invalid when dataset split, evaluation conditions, or multiple uncontrolled factors change, and distinguish a primary decision metric from diagnostic metrics.

[Campus explanation, task and post-attempt feedback](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/sprint-2/02-controls-confounders-and-decision-metrics/lesson.md).


## A-S2-03 · Building the five-run experiment matrix

Choose five distinct positive alphas including the alpha=1 anchor before executing. Larger alpha restrains learned weights more strongly; excessive restraint can miss useful patterns. Explain the spacing, name fixed conditions and keep a changed-partition example outside the five-run plan. Edit configs/experiment.json and retain your hypothesis; this plan cell does not train.

### Learning outcome

Define a five-run experiment matrix for the provided training and inference pipeline starter that holds the dataset split and evaluation conditions constant, names the factor varied in each comparison, selects a primary metric tied to the inference brief, and labels any run that is not directly comparable.

[Campus explanation, task and post-attempt feedback](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/sprint-2/03-building-the-five-run-experiment-matrix/lesson.md).


In [ ]:
plan = json.loads((ROOT / 'configs/experiment.json').read_text())
assert (plan['primary_metric'], plan['direction'], plan['partition']) == ('rmse_kwh', 'minimize', 'validation')
assert len(plan['alphas']) == 5 and len(set(plan['alphas'])) == 5 and 1 in plan['alphas'], 'This investigation needs five strengths including its logging anchor.'
print('Declared alphas:', progress.plan(plan['alphas']))

## A-S2-04 · Revisiting MLflow in a pipeline context

Execute the declared matrix once and retain its five IDs. Inspect one actual record and connect each field to Workspace.run: before-fit automatic capture, explicit validation logging, deliberate model/recovery writes and finally hook cleanup. Project reopens decisions; Workspace opens the tracking store. The optional notebook uses the same core as the primary VS Code script. Start the local UI in a separate terminal; it stays running. Compare the full ID in Python and UI.

### Learning outcome

Use the provided MLflow setup to log parameters, metrics, and artifacts automatically as part of the cumulative pipeline, building on basic experiment logging from T4 (AI track).

[Campus explanation, task and post-attempt feedback](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/sprint-2/04-revisiting-mlflow-in-a-pipeline-context/lesson.md).


In [ ]:
if not progress.data.get('matrix_run_ids'):
    records = [ws.run(Config(alpha=a, range_guard=True, autolog=True), name=f'alpha-{a}')
               for a in progress.require('planned_alphas')]
    progress.record_matrix(ws, records)
rows = progress.matrix(ws)
rows.to_csv(state / 'runs.csv', index=False)
display(rows.sort_values('alpha'))
print('Saved IDs:', progress.require('matrix_run_ids'))

Inspect one actual record next. A parameter answers what ran; `rmse_kwh` answers held-out prediction error; `reproduction/manifest.json` connects recipe and environment. The model URI identifies the fitted artifact. Look inside `src/linden_mlops/lifecycle.py` to find the calls that logged these fields.

In [ ]:
sample_id = progress.require('matrix_run_ids')[0]
raw = ws.client.get_run(sample_id)
print('Run:', sample_id)
print('Parameters:', raw.data.params)
print('Metrics:', raw.data.metrics)
print('Artifacts:', [a.path for a in ws.client.list_artifacts(sample_id)])
print('Recovery files:', [a.path for a in ws.client.list_artifacts(sample_id, 'reproduction')])
print('Local UI command: uv run linden-mlops ui --state', ws.path)

## A-S2-05 · Comparing multiple runs systematically

Use the actual MLflow UI to select only the five saved runs, inspect controls and compare validation rmse_kwh. The default graph may show a training diagnostic; choose the primary metric explicitly. Check raw precision before choosing and save the actual run ID. A plot alone does not satisfy UI comparison. Complete the selection attempt before executing its check or the solved reference figure below.

### Learning outcome

Use the MLflow UI to compare 5+ experiment runs against each other and identify the best-performing configuration for a given metric.

[Campus explanation, task and post-attempt feedback](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/sprint-2/05-comparing-multiple-runs-systematically/lesson.md).


In [ ]:
# Your choice: progress.select(ws, 'ACTUAL_RUN_ID')
# Explain the comparable controls and why this metric supports your choice.

In [ ]:
selection = progress.selected()
print('Retained selection:', selection)
(state / 'selected-config.json').write_text(json.dumps(selection['config'], indent=2) + '\n')

### Read the comparable metric plot

Both vertical axes are in kWh, but RMSE and MAE answer different error questions. Alpha uses a logarithmic horizontal axis; each point is an observed run, with no implied intermediate fit. The highlighted winner follows the raw validation RMSE rule, not a training diagnostic or final score. A small winning margin is not evidence of statistical significance.


In [ ]:
experiment_figure = plot_experiments(rows)
display(experiment_figure)
print(experiment_figure.linden_data)
experiment_paths = save_figure(experiment_figure, figures / 'experiment-comparison__01__en')


## A-S2-06 · Autologging and reducing manual tracking overhead

A hook is recording code triggered when fitting runs. Reuse the saved autologged alpha=1 anchor and create one otherwise identical manual run, keeping guard and validation settings. Predict a field that should appear only with autologging; compare observed keys, not just flags. Save logging_pair and reuse it. A permitted score difference of 0.000001 kWh (1e-6) checks that logging alone did not change evaluation. Keep the key comparison and persistence branch your own work.

### Learning outcome

Configure autologging for the pipeline and explain the tradeoff between autologging convenience and control over what gets logged.

[Campus task and post-attempt feedback](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/sprint-2/06-autologging-and-reducing-manual-tracking-overhead/lesson.md).


In [ ]:
# Your attempt: create one Config(alpha=1, range_guard=True, autolog=False) run.
# Keep the existing autologged anchor; save both IDs in progress.data["logging_pair"] via progress.save.
# Compare run.data.params, run.data.metrics and artifacts through ws.client.
# Predict fields first and verify held-out RMSE agrees within 1e-6 kWh.


### Compare after configuring the logging pair

<details><summary>Open after your own logging comparison</summary>

**Result you should get:** the autologged record contains an estimator field such as `ridge__alpha` absent from the manual record. Both keep explicit validation `rmse_kwh` and one explicitly logged model. Validation RMSE agrees within `1e-6` kWh.

**How:** compare sets of actual keys, not just flags. Enable the hook before fitting; it cannot retroactively observe a completed fit. A training diagnostic does not replace the declared held-out metric. Inspect the Campus feedback if predictions differ before attributing the difference to logging.

</details>


## A-S2-07 · Validating an assistant-generated experiment summary

Give an institution-approved assistant only the five comparable raw rows. Ask for a full run ID using lower validation rmse_kwh, recorded controls and limits. Keep the actual request/input/response. Check the numeric nomination with verify_summary, then audit every prose claim separately. The deliberate worst-run example must return False. A supplied authored response is practice, not proof of an actual request.

### Learning outcome

Direct an approved assistant to generate a summary comparison across logged experiment runs, validate every recommendation against the raw MLflow data and the declared primary metric, and flag any inaccuracy or invalid comparison.

[Campus explanation, task and post-attempt feedback](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/sprint-2/07-validating-an-assistant-generated-experiment-summary/lesson.md).


In [ ]:
worst_id = rows.sort_values('rmse_kwh').iloc[-1].run_id
wrong = verify_summary(rows, worst_id)
print('Deliberately wrong advice:', wrong)
assert not wrong['recommendation_matches_metric']

In [ ]:
# Add the actual response's proposed ID and run verify_summary(rows, proposed_id).
# Keep one claim -> raw field -> supported/inaccurate/unsupported table.
# Qualify only actual issues; the response may be fully accurate.

## A-S2-08 · Experiment tracking deliverable

Harbor supplies unfamiliar observations and a client note assembled from retained runs. An observation is a reported measurement; a run is an execution; a configuration is its settings. Determine their relationships yourself. Reuse the five configurations, UI record, logging pair and actual assistant response. Prepare once, preserve the packet and audit its claims. Keep the actual final holdout unopened. The supplied client note is separate authored exercise text.

### Learning outcome

Deliverable: at least 5 experiment configurations logged and compared in MLflow, an autologging setup, and an AI-assisted summary recommending which configuration to promote - with the summary validated against raw data.

[Campus task and post-attempt feedback](https://github.com/richhiey/ml-llm-ops/blob/main/docs/course/sprint-2/08-experiment-tracking-deliverable/lesson.md).


In [ ]:
comparison_packet = state / 'harbor-comparison'
if not comparison_packet.exists():
    # This external course tool supplies constructed incoming evidence; it is not an app command.
    prepared = subprocess.run([
        sys.executable, 'docs/course/exercises/prepare-evidence.py', 'comparison',
        '--destination', str(comparison_packet), '--work-dir', str(state)],
        cwd=ROOT, capture_output=True, text=True)
    print(prepared.stdout)
    if prepared.returncode:
        raise RuntimeError(prepared.stderr)
observations = json.loads((comparison_packet / 'observations.json').read_text())
display(pd.DataFrame(observations['observations']).drop(columns='controls'))
print((comparison_packet / 'client-note.txt').read_text())


In [ ]:
# Your attempt: group observations by actual run_id and inspect metric/evaluation basis.
# Audit the note against raw fields, retain eligible validation IDs, and justify the rank.
# Explain cost/readiness evidence that the table cannot supply. Do not open final rows.


### Compare after auditing Harbor's recommendation

<details><summary>Open after your observation/run audit</summary>

**Result you should get:** six labels describe five actual matrix run IDs. The repeated run's training-root-mean-squared-error observation is not another validation candidate. Rank only the five comparable validation `rmse_kwh` observations and retain your actual selected ID. No cost or readiness conclusion follows from that ranking.

**How:** separate measurement identity, execution identity and evaluation basis. Cross-check values/controls against raw MLflow records. Your actual assistant record remains a separate evidence item; a prepared client note cannot substitute for the required request.

</details>
